In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies= pd.read_csv("tmdb_5000_movies.csv")
credits=pd.read_csv("tmdb_5000_credits.csv")

In [4]:
movies=movies.merge(credits, on='title')

In [6]:
movies=movies[['movie_id','title','genres','keywords','overview','cast','crew']]

In [ ]:
movies.isnull().sum()

In [8]:
movies.dropna(inplace=True)

In [9]:
import ast
def convert(obj):
    l=[]
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [10]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [12]:
def convert3(obj):
    l=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter!=3:
            l.append(i['character'])
            counter+=1
        else:
            break
    return l

In [13]:
movies['cast']=movies['cast'].apply(convert3)

In [14]:
import ast

def fetch_director(obj):
    if isinstance(obj, list):
        data = obj
    else:
        try:
            data = ast.literal_eval(obj)
        except:
            return []

    for i in data:
        if i.get('job') == 'Director':
            return [i.get('name')]
    
    return []

In [15]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [17]:
movies['overview'] = movies['overview'].astype(str).apply(lambda x: x.split())

In [18]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [19]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [21]:
new_df=movies[['movie_id','title','tags']]

In [23]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

In [24]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [ ]:
!pip install nltk

In [26]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [27]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [28]:
new_df['tags']=new_df['tags'].apply(stem)

In [29]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [30]:
vectors=cv.fit_transform(new_df['tags']).toarray()

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
similarity=cosine_similarity(vectors)

In [33]:
def recommend(movie):
    movie_index=new_df[new_df['title']==movie].index[0]
    distance=similarity[movie_index]
    movies_list=sorted(list(enumerate(distance)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
        
    
    return

In [36]:
import pickle

pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))